In [3]:
"""
You can think of this like a "sliding window"
| Shift (s) | Window (T[s:s+4]) |
| ----------| ----------------|
| 0         | T[0:4]          |
| 1         | T[1:5]          |
| 2         | T[2:6]          |
| 3         | T[3:7]          |
| 4         | T[4:8]          |
"""
def naive_string_matcher(T, P):
    n = len(T)
    m = len(P)

    for s in range(n - m + 1):
        if T[s:s + m] == P:
            print("Pattern occurs with shift", s)

In [4]:
T = "AABAACAADAABAABA"
P = "AABA"

naive_string_matcher(T, P)

Pattern occurs with shift 0
Pattern occurs with shift 9
Pattern occurs with shift 12


In [5]:
"""
With P distinct characters:

Every mismatch causes us to skip at least one new character.
If we matched j characters, we skip exactly j positions.
Therefore, each text character is involved in only a constant number of comparisons.
"""
def distinct_naive_matcher(T, P):
    n = len(T)
    m = len(P)

    s = 0

    while s <= n - m:
        j = 0

        while j < m and T[s + j] == P[j]:
            j += 1

        if j == m:
            print("Pattern occurs at shift", s)
            s += m          # all characters are distinct
        elif j == 0:
            s += 1          # first character mismatched
        else:
            s += j          # skip the matched characters

In [6]:
def gap_match(T, P):
    pieces = P.split("<>")
    pos = 0

    for piece in pieces:
        i = T.find(piece, pos)

        if i == -1:
            return False

        pos = i + len(piece)

    return True

In [8]:
"""
Rabin-Karp String Matching

T : text
P : pattern
d : number of possible characters (alphabet size)
q : prime number used for hashing
"""
def rabin_karp_matcher(T, P, d=256, q=101):
    n = len(T)
    m = len(P)

    # d^(m-1) % q
    h = pow(d, m - 1, q)

    p = 0          # hash value of pattern
    t = 0          # hash value of current text window

    # Compute initial hashes
    for i in range(m):
        p = (d * p + ord(P[i])) % q
        t = (d * t + ord(T[i])) % q

    # Slide pattern over text
    for s in range(n - m + 1):
        
        # If hashes match, compare actual strings
        if p == t:
            if P == T[s:s + m]:
                print(f"Pattern occurs at shift {s}")

        # Compute next window hash
        if s < n - m:
            t = (d * (t - ord(T[s]) * h) + ord(T[s + m])) % q

            # Make hash positive
            if t < 0:
                t += q

In [9]:
"""
text = "ABCCDDAEFG"
pattern = "CDD"

Instead of comparing
CDD

against
ABC
BCC
CCD
CDD
DDA
...

character by character every time,
it compares hash values.
Think of a hash as a fingerprint.

"CDD"  ---> hash = 54

"ABC"  ---> hash = 19

"BCC"  ---> hash = 88

"CCD"  ---> hash = 91

"CDD"  ---> hash = 54  <-- Match!

Only when the fingerprints match do we actually compare the strings.


Breaking down the formula
t = (d * (t - ord(T[s]) * h) + ord(T[s + m])) % q

Think of it as four small steps:

Step 1 Remove first character

t - ord(T[s]) * h

Visual ABC

remove A

BC

Step 2 Shift remaining characters

d * (...)

Visual

BC

↓

BC_

Multiplying by d is like shifting the characters one position to the left in a base-d number system.

Step 3 Append new character

+ ord(T[s+m])

Visual

BC_

↓

BCC

Step 4 Keep the number small

% q

Otherwise the hash grows extremely large.

Step 5 — Compare hashes

Suppose

Pattern hash = 82

Window hashes

ABC →59

BCC →74

CCD →31

CDD →82

Hashes match.

Now we verify

if P == T[s:s+m]:
CDD

CDD

Equal!

Output

Pattern occurs at shift 3
"""

'\ntext = "ABCCDDAEFG"\npattern = "CDD"\n\nInstead of comparing\nCDD\n\nagainst\nABC\nBCC\nCCD\nCDD\nDDA\n...\n\ncharacter by character every time,\nit compares hash values.\nThink of a hash as a fingerprint.\n\n"CDD"  ---> hash = 54\n\n"ABC"  ---> hash = 19\n\n"BCC"  ---> hash = 88\n\n"CCD"  ---> hash = 91\n\n"CDD"  ---> hash = 54  <-- Match!\n\nOnly when the fingerprints match do we actually compare the strings.\n'